In [1]:
import os
import json
import hashlib
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def load_json(path):
    return json.load(open(path , 'r', encoding='utf-8'))

c:\Users\ASUS\anaconda3\envs\finpros\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
daiphatthanh = load_json(r'post\daiphatthanh_100_posts.json')
theanh = load_json(r'post\theanh28_100_posts.json')
day1000 = load_json(r'post\day_1000_100_posts.json')

In [3]:
def get_urls(posts):
    urls = []
    for post in posts:
        urls.append(post['url'])

    return urls

In [4]:
daiphatthanh_urls = get_urls(daiphatthanh)
theanh_urls = get_urls(theanh)
day1000_urls = get_urls(day1000)

In [5]:
def _safe_text(item):
    if isinstance(item, str):
        return item.strip()

    if isinstance(item, dict):
        value = item.get('text', '')
        if isinstance(value, dict):
            value = value.get('text', '') or value.get('value', '')
        if isinstance(value, str):
            tags = item.get('tags', [])
            cleaned_text = value

            if isinstance(tags, list):
                for tag in tags:
                    tag_text = ''
                    if isinstance(tag, dict):
                        tag_text = tag.get('text', '')
                    elif isinstance(tag, str):
                        tag_text = tag

                    if isinstance(tag_text, str) and tag_text:
                        cleaned_text = cleaned_text.replace(tag_text, ' ')

            return ' '.join(cleaned_text.split()).strip()

    return ''


def get_comments(post_url, client_id='client_51gm9xxri_100009489619122', comment_type='CHRONOLOGICAL_UNFILTERED_INTENT_V1', api_url='https://api.fbaio.xyz/call'):
    headers = {'Content-Type': 'application/json'}
    all_comment_texts = []
    cursor = ''

    while True:
        payload = {
            'id': client_id,
            'apiname': 'get_list_fb_comment',
            'apiparams': {
                'url': post_url,
                'type': comment_type,
                'cursor': cursor
            }
        }

        response = requests.post(api_url, json=payload, headers=headers)
        response.raise_for_status()

        result = response.json().get('result', {})
        comments = result.get('comments', [])
        for comment in comments:
            text = _safe_text(comment)
            if text:
                all_comment_texts.append(text)

        page_info = result.get('pageInfo', {})
        has_next_page = page_info.get('hasNextPage', False)
        next_cursor = page_info.get('endCursor')

        if not has_next_page:
            break
        if not next_cursor or next_cursor == cursor:
            break

        cursor = next_cursor

    return all_comment_texts

In [6]:
def _resume_key(urls):
    return hashlib.md5('||'.join(urls).encode('utf-8')).hexdigest()


def _load_resume_state(resume_path, urls):
    if not resume_path or not os.path.exists(resume_path):
        return {}

    try:
        with open(resume_path, 'r', encoding='utf-8') as f:
            state = json.load(f)
    except Exception:
        return {}

    if state.get('key') != _resume_key(urls):
        return {}

    results = state.get('results', {})
    if not isinstance(results, dict):
        return {}

    return results


def _save_resume_state(resume_path, urls, results_by_idx):
    if not resume_path:
        return

    os.makedirs(os.path.dirname(resume_path) or '.', exist_ok=True)

    ordered_results = [results_by_idx.get(str(i), []) for i in range(len(urls))]
    flattened = []
    for comments in ordered_results:
        flattened.extend(comments)

    results_by_url = []
    for idx, url in enumerate(urls):
        results_by_url.append({
            'index': idx,
            'url': url,
            'comments': ordered_results[idx]
        })

    state = {
        'key': _resume_key(urls),
        'total_urls': len(urls),
        'completed_urls': len(results_by_idx),
        'results': results_by_idx,
        'results_by_url': results_by_url,
        'comments': flattened
    }
    with open(resume_path, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False)


def process_page(urls, show_progress=True, max_workers=6, resume=True, resume_path='comment/process_page_output.json', checkpoint_every=5, flatten=True):
    if not urls:
        return []

    results_by_idx = _load_resume_state(resume_path, urls) if resume else {}
    done_indices = {int(k) for k in results_by_idx.keys() if str(k).isdigit()}
    pending = [(idx, url) for idx, url in enumerate(urls) if idx not in done_indices]

    progress = tqdm(total=len(urls), initial=len(done_indices), desc='Crawling comments', unit='post', disable=not show_progress)

    if pending:
        worker_count = max(1, min(max_workers, len(pending)))
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            future_map = {executor.submit(get_comments, url): idx for idx, url in pending}
            completed_since_save = 0

            for future in as_completed(future_map):
                idx = future_map[future]
                try:
                    results_by_idx[str(idx)] = future.result()
                except Exception as ex:
                    results_by_idx[str(idx)] = []
                    print(f'Failed at index {idx}: {ex}')

                progress.update(1)
                completed_since_save += 1

                if resume and completed_since_save >= max(1, checkpoint_every):
                    _save_resume_state(resume_path, urls, results_by_idx)
                    completed_since_save = 0

            if resume:
                _save_resume_state(resume_path, urls, results_by_idx)

    progress.close()

    ordered_results = [results_by_idx.get(str(i), []) for i in range(len(urls))]
    if not flatten:
        return ordered_results

    flattened = []
    for comments in ordered_results:
        flattened.extend(comments)
    return flattened

In [7]:
# daiphatthanh_comments = process_page(
#     daiphatthanh_urls,
#     max_workers=4,
#     resume=True,
#     resume_path='comment/daiphatthanh_comments.json',
#     show_progress=True,
#     flatten=False
# )

In [9]:
theanh_comments = process_page(
    theanh_urls,
    max_workers=4,
    resume=True,
    resume_path='comment/theanh_comments.json',
    show_progress=True,
    flatten=True
)

Crawling comments:  80%|███████▉  | 86/108 [10:54<04:24, 12.03s/post]

Failed at index 51: 504 Server Error: Gateway Timeout for url: https://api.fbaio.xyz/call
Failed at index 65: 504 Server Error: Gateway Timeout for url: https://api.fbaio.xyz/call


Crawling comments:  81%|████████  | 87/108 [10:56<03:06,  8.90s/post]

Failed at index 20: 504 Server Error: Gateway Timeout for url: https://api.fbaio.xyz/call


Crawling comments: 100%|██████████| 108/108 [13:34<00:00,  7.54s/post]


In [11]:
day1000_urls[:3]

['https://www.facebook.com/d1wts/posts/pfbid0ZjvvhRpoehDAokYjtW4M8gMf3oAXgwXLajkGABqMn7E15bELxwGuDWfGoxQg8Mz4l',
 'https://www.facebook.com/d1wts/posts/pfbid0NZmZBq7VccZ5cjUrygdgkc8qEVSLSQREc29QGPj5KAZn91dDDLECHmG1WMwsnnAwl',
 'https://www.facebook.com/d1wts/posts/pfbid0V2uV7zdLZoRvDFnLo6jS5TFkCUp2NrUBKXmYAbk2VoQgyzYftjCKAUs3MLbXjbPsl']

In [12]:
day1000_comments = process_page(
    day1000_urls,
    max_workers=4,
    resume=True,
    resume_path='comment/day1000_comments.json',
    show_progress=True,
    flatten=True
)

Crawling comments: 100%|██████████| 108/108 [08:25<00:00,  4.68s/post]
